# 14 — HuggingFace Integration (Guarded)

**Purpose:** Exercise TorchLens's HuggingFace-specific surfaces: `autoroute` input detection
and routing, `tl.bridge.hf` adapter helpers, and attention facets on real transformer models.

**Soft guard:** If `transformers` is not installed or the cached tiny models are unavailable,
this notebook skips cleanly. On this machine it should run all sections with real output.

**Surfaces covered:**
- [ ] `torchlens.autoroute.input` — registry: `.list()`, `.info()`, `.register()`, `.unregister()`, `.snapshot()`
- [ ] `tl.trace(hf_model, raw_text)` — autoroute detects text and routes through bridge
- [ ] `tl.bridge.hf.trace_text(model, text, tokenizer=...)` — explicit bridge call
- [ ] `tl.bridge.hf.trace_image(model, image)` — image bridge (skipped, no PIL)
- [ ] `Trace.input_preprocessor` — `ResolvedPreprocessing` provenance record
- [ ] Full `tl.trace` on a tiny HF text model — `Trace` repr, `summary()`, `layer_labels`
- [ ] `save=tl.func(...)` selective capture on transformer layers
- [ ] `tl.facets.list()` — built-in recipe registry
- [ ] `trace.modules[path].facets` — `FacetView` with `.keys()`, `.menu()`, `.get()`
- [ ] `FacetView.get('q'/'k'/'v')` — attention head tensors as `Facet` objects
- [ ] `Facet.value` — underlying tensor, `Facet.spec` — provenance
- [ ] `FacetView.head(i)` — `AttentionHeadView` with `.q/.k/.v`
- [ ] `FacetView.get('n_heads')`, `get('d_head')` — structural metadata
- [ ] ViT tiny model trace (vision transformer)
- [ ] `trace.draw(vis_save_only=True, vis_fileformat='pdf')` on HF model (text confirm)

## 0. Soft guard + environment

In [ ]:
import pathlib
import sys
import warnings

# Ensure _models.py is importable (same pattern as template 00)
sys.path.insert(0, str(pathlib.Path.cwd()))

import torch
import torchlens as tl
import torchlens.autoroute as ar

# --- Soft guard ---
_SKIP = False
_SKIP_REASON = ""

try:
    import transformers  # noqa: F401
except ImportError:
    _SKIP = True
    _SKIP_REASON = "transformers not installed -- pip install transformers to run this notebook"

if not _SKIP:
    # Verify a tiny cached model exists
    _CACHE_ROOT = pathlib.Path.home() / ".cache" / "huggingface" / "hub"
    _TINY_BERT = "models--hf-internal-testing--tiny-random-distilbert"
    if not (_CACHE_ROOT / _TINY_BERT).exists():
        _SKIP = True
        _SKIP_REASON = (
            "tiny-random-distilbert not cached; run: "
            "from transformers import AutoModel; "
            "AutoModel.from_pretrained('hf-internal-testing/tiny-random-distilbert')"
        )

if _SKIP:
    print(f"SKIP: {_SKIP_REASON}")
    print("All subsequent cells are no-ops; this notebook exits green.")
else:
    print(
        f"torchlens {tl.__version__} | torch {torch.__version__} | transformers {transformers.__version__}"
    )
    print("Guard: OK -- all sections will run with real HF model output.")

## 1. `torchlens.autoroute.input` — the detector registry

TorchLens ships built-in input detectors for HuggingFace payloads. The registry
resolves which detector claims a given `(model, payload)` pair — lower `priority`
values run first.

In [ ]:
if _SKIP:
    print("SKIP")
else:
    print("=== torchlens.autoroute.input: built-in detector registry ===")
    for det in ar.input.list():
        info = ar.input.info(det.name)
        print(f"  name={det.name!r:25s}  priority={det.priority}")
        print(f"    doc: {info['doc'].splitlines()[0][:80]}")
    print()

    # --- register / unregister API ---
    print("=== Custom detector registration ===")

    @ar.input.register(name="demo_custom", priority=5)
    def _demo_detector(model, payload, **kwargs):
        """Example: never claims dispatch, just for illustration."""
        return None  # pass-through

    print("After register: detector order =", [d.name for d in ar.input.list()])

    ar.input.unregister("demo_custom")
    print("After unregister:", [d.name for d in ar.input.list()])
    print()

    # --- snapshot context manager ---
    print("=== snapshot() context manager ===")
    with ar.input.snapshot():
        ar.input.unregister("hf_text")
        print("  inside snapshot (hf_text removed):", [d.name for d in ar.input.list()])
    print("  after snapshot (restored):", [d.name for d in ar.input.list()])

## 2. `tl.trace(hf_model, raw_text)` — autoroute in action

When `tl.trace` receives a HuggingFace model and a raw string, the `hf_text`
detector claims the dispatch, tokenizes automatically, and returns a normal `Trace`.
No manual tokenizer call needed.

In [ ]:
if _SKIP:
    print("SKIP")
else:
    from transformers import AutoModel, AutoTokenizer

    _BERT_ID = "hf-internal-testing/tiny-random-distilbert"

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _bert = AutoModel.from_pretrained(_BERT_ID)

    print(f"Model: {_bert.__class__.__name__}  ({_BERT_ID})")
    print(f"model.config.name_or_path: {_bert.config.name_or_path!r}")
    print()

    # Autoroute: pass raw text -- hf_text detector fires
    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _bert_trace = tl.trace(_bert, "The quick brown fox")

    print("=== tl.trace(model, raw_string) ===  (autoroute claims via hf_text detector)")
    print(repr(_bert_trace))
    print()
    print("Trace.input_preprocessor (provenance injected by bridge):")
    print(_bert_trace.input_preprocessor)
    print()
    print(f"Total layers: {len(_bert_trace.layer_labels)}")
    print("First 12 layer labels:", _bert_trace.layer_labels[:12])
    print("Last 5 layer labels: ", _bert_trace.layer_labels[-5:])

## 3. `Trace.summary()` on a real transformer

The summary now shows the `Input preprocessing:` block with the resolved tokenizer provenance.

In [ ]:
if _SKIP:
    print("SKIP")
else:
    print(_bert_trace.summary())

## 4. `tl.bridge.hf.trace_text` — explicit bridge call

`tl.bridge.hf.trace_text(model, text, tokenizer=...)` is the lower-level helper that
the `hf_text` autoroute detector delegates to. Useful when you need to pass an
explicit tokenizer rather than relying on auto-resolution.

In [ ]:
if _SKIP:
    print("SKIP")
else:
    from torchlens.bridge import hf as tl_hf

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _tokenizer = AutoTokenizer.from_pretrained(_BERT_ID)

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _explicit_trace = tl_hf.trace_text(
            _bert,
            "Explicit tokenizer call",
            tokenizer=_tokenizer,
        )

    print("=== tl.bridge.hf.trace_text(model, text, tokenizer=tok) ===")
    print(repr(_explicit_trace))
    print()
    print("input_preprocessor.source:", _explicit_trace.input_preprocessor.source)
    print("input_preprocessor.identifier:", _explicit_trace.input_preprocessor.identifier)
    print("input_preprocessor.config:")
    for k, v in _explicit_trace.input_preprocessor.config.items():
        print(f"  {k}: {v!r}")

## 5. Selective capture on transformer layers: `save=tl.func(...)`

Use the unified predicate surface to capture only attention-related ops from the
transformer. `scaleddotproductattention` is the fused SDPA kernel PyTorch uses for
modern HF models (SDPA attention backend).

In [ ]:
if _SKIP:
    print("SKIP")
else:
    # Capture only SDPA ops + linear projections inside attention
    _attn_pred = tl.func("scaleddotproductattention")
    _linear_pred = tl.func("linear") & tl.in_module("attention")

    with warnings.catch_warnings():
        warnings.simplefilter("ignore")
        _selective = tl.trace(
            _bert,
            "Selective attention capture",
            save=_attn_pred | _linear_pred,
        )

    sdpa_ops = [lbl for lbl in _selective.layer_labels if "scaleddotproduct" in lbl]
    linear_attn_ops = [lbl for lbl in _selective.layer_labels if "linear" in lbl]

    print(f"Total layers captured: {len(_selective.layer_labels)}")
    print(f"SDPA layers ({len(sdpa_ops)}): {sdpa_ops}")
    print(f"Linear-in-attention layers ({len(linear_attn_ops)}): {linear_attn_ops[:8]}")
    print()

    # Inspect one SDPA op
    _sdpa0 = _selective[sdpa_ops[0]]
    print(f"SDPA op repr ({sdpa_ops[0]}):")
    print(repr(_sdpa0))

## 6. `tl.facets.list()` — built-in semantic recipe registry

TorchLens ships built-in facet recipes for the major HuggingFace attention implementations.
A recipe maps a module class name to named semantic views (q, k, v, attention pattern, etc.).

In [ ]:
if _SKIP:
    print("SKIP")
else:
    _recipes = tl.facets.list()
    print(f"{len(_recipes)} built-in facet recipes:")
    for r in _recipes:
        cls_str = ", ".join(r.class_names[:3])
        if len(r.class_names) > 3:
            cls_str += f", +{len(r.class_names) - 3} more"
        print(f"  {r.recipe_name!r:30s}  classes=({cls_str})")

## 7. Attention facets — `FacetView.get('q'/'k'/'v')` and `FacetView.head(i)`

After a trace, every module exposes a `FacetView` via `.facets`. For HF attention modules,
the built-in recipe populates `q`, `k`, `v`, `attn_out`, and per-head structural metadata.

In [ ]:
if _SKIP:
    print("SKIP")
else:
    # Use the full trace (not the selective one) so facets have all ops available
    _attn_path = "transformer.layer.0.attention"
    _attn_mod = _bert_trace.modules[_attn_path]

    print(f"Module: {_attn_path!r}")
    print(repr(_attn_mod))
    print()

    _fv = _attn_mod.facets
    print("FacetView keys:", _fv.keys())
    print()

    # Structural metadata (plain scalars)
    print(f"n_heads   = {_fv.get('n_heads')}")
    print(f"d_head    = {_fv.get('d_head')}")
    print()

    # q, k, v Facet objects
    _q = _fv.get("q")
    _k = _fv.get("k")
    _v = _fv.get("v")
    print(f"q facet type: {type(_q).__name__}  value shape: {_q.value.shape}")
    print(f"k facet type: {type(_k).__name__}  value shape: {_k.value.shape}")
    print(f"v facet type: {type(_v).__name__}  value shape: {_v.value.shape}")
    print("(shape: batch x seq_len x n_heads x d_head)")
    print()

    # Facet.spec — provenance
    print("q.spec (provenance):")
    _spec = _q.spec
    print(f"  home_label:       {_spec.home_label!r}")
    print(f"  home_address:     {_spec.home_address!r}")
    print(f"  recipe_id:        {_spec.recipe_id!r}")
    print(f"  capability_class: {_spec.capability_class!r}")
    print(f"  transforms:       {_spec.transforms}")
    print(
        f"  capability_flags: read={_spec.capability_flags.read}  "
        f"grad={_spec.capability_flags.grad}  write={_spec.capability_flags.write}"
    )

In [ ]:
if _SKIP:
    print("SKIP")
else:
    # Per-head sliced view via FacetView.head(i)
    n_heads = _fv.get("n_heads")
    print(f"=== FacetView.head(i) -- sliced view for each of {n_heads} heads ===")
    for i in range(n_heads):
        _hv = _fv.head(i)
        print(
            f"  head {i}: type={type(_hv).__name__}  "
            f"q.value.shape={_hv.q.value.shape}  "
            f"k.value.shape={_hv.k.value.shape}  "
            f"v.value.shape={_hv.v.value.shape}"
        )

## 8. `FacetView.menu()` — what facets are available vs. need recapture

The `menu()` tells you which facets are `'available_now'` from the current trace,
and which need `reconstruction_ready=True` or `save_arg_values=True` to unlock.
This is how a user discovers what they can and cannot inspect.

In [ ]:
if _SKIP:
    print("SKIP")
else:
    _menu = _fv.menu()
    print("FacetView.menu() for DistilBertSdpaAttention (layer 0):")
    for key, item in _menu.items():
        hint = f"  hint: {item.save_hint}" if item.save_hint else ""
        print(f"  {key!r:15s}  status={item.status!r}{hint}")

## 9. Vision transformer (ViT) trace

`hf-internal-testing/tiny-random-vit` is a 5-layer ViT with 30x30 images cached
locally. Its attention modules use `ViTSelfAttention` (a different recipe than DistilBert).

In [ ]:
if _SKIP:
    print("SKIP")
else:
    _VIT_ID = "hf-internal-testing/tiny-random-vit"
    _vit_cache = _CACHE_ROOT / "models--hf-internal-testing--tiny-random-vit"

    if not _vit_cache.exists():
        print(f"SKIP ViT section: {_VIT_ID} not cached.")
        _vit_trace = None
    else:
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            _vit = AutoModel.from_pretrained(_VIT_ID)

        _cfg = _vit.config
        _H, _W = _cfg.image_size, _cfg.image_size
        _C = _cfg.num_channels
        _pixel_values = torch.randn(1, _C, _H, _W)

        print(f"Model: {_vit.__class__.__name__}  image_size={_H}x{_W}  channels={_C}")
        print(f"Input tensor: {_pixel_values.shape}")
        print()

        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            _vit_trace = tl.trace(_vit, _pixel_values)

        print(repr(_vit_trace))
        print()
        print("Layer count:", len(_vit_trace.layer_labels))
        print("Modules with 'attention' in path (sample):")
        # Use .keys() -- the stable ModuleAccessor accessor
        _attn_mod_paths = [
            k for k in _vit_trace.modules.keys() if "attention" in k.lower() and ":" not in k
        ]
        for m_path in _attn_mod_paths[:8]:
            print(f"  {m_path!r}")

In [ ]:
if _SKIP or _vit_trace is None:
    print("SKIP")
else:
    # ViTSelfAttention facets
    _vit_attn_path = "encoder.layer.0.attention.attention"
    _vit_attn_mod = _vit_trace.modules[_vit_attn_path]
    print(f"Module: {_vit_attn_path!r}")
    print(repr(_vit_attn_mod))
    print()

    _vit_fv = _vit_attn_mod.facets
    print("FacetView keys:", _vit_fv.keys())

    _vit_n_heads = _vit_fv.get("n_heads")
    _vit_d_head = _vit_fv.get("d_head")
    print(f"n_heads={_vit_n_heads}  d_head={_vit_d_head}")
    print()

    _vit_q = _vit_fv.get("q")
    _vit_k = _vit_fv.get("k")
    _vit_v = _vit_fv.get("v")
    print(f"q shape: {_vit_q.value.shape}  (batch, patches+cls, n_heads, d_head)")
    print(f"k shape: {_vit_k.value.shape}")
    print(f"v shape: {_vit_v.value.shape}")
    print()

    # Per-head view
    _vit_h0 = _vit_fv.head(0)
    print(f"head(0).q.value.shape: {_vit_h0.q.value.shape}  (batch, patches+cls, d_head)")

## 10. `trace.draw()` on a real transformer — text-only output

TEXT OUTPUT ONLY: render to a temp PDF and confirm size. Embedding graph PNGs in the
notebook inflates the file and trips detect-secrets hooks.

In [ ]:
if _SKIP:
    print("SKIP")
else:
    import os
    import tempfile

    with tempfile.TemporaryDirectory() as _tmp:
        _out = os.path.join(_tmp, "bert_graph")
        try:
            with warnings.catch_warnings():
                warnings.simplefilter("ignore")
                _bert_trace.draw(
                    vis_outpath=_out,
                    vis_save_only=True,
                    vis_fileformat="pdf",
                    vis_mode="rolled",  # rolled = compact for large transformer
                )
            _pdf = pathlib.Path(_out + ".pdf")
            _size_kb = _pdf.stat().st_size / 1024
            print(f"draw() rolled PDF: {_pdf.name}  size={_size_kb:.1f} KB  OK")
        except Exception as _e:
            print(f"draw() error: {_e}")

    if _vit_trace is not None:
        with tempfile.TemporaryDirectory() as _tmp:
            _out = os.path.join(_tmp, "vit_graph")
            try:
                with warnings.catch_warnings():
                    warnings.simplefilter("ignore")
                    _vit_trace.draw(
                        vis_outpath=_out,
                        vis_save_only=True,
                        vis_fileformat="pdf",
                        vis_mode="rolled",
                    )
                _pdf = pathlib.Path(_out + ".pdf")
                print(
                    f"ViT draw() rolled PDF: {_pdf.name}  size={_pdf.stat().st_size / 1024:.1f} KB  OK"
                )
            except Exception as _e:
                print(f"ViT draw() error: {_e}")

---

## ⚠️ GAPs / ergonomic smells

- **`tl.bridge.hf` is not in `tl.__all__`** and not accessible directly from the top-level
  `tl` namespace as `tl.bridge.hf`. Users must do `from torchlens.bridge import hf as tl_hf`
  or `import torchlens.bridge.hf` explicitly. The autoroute detector handles the common case,
  but if users want to call `trace_text`/`trace_image` directly, the import path is not
  surfaced anywhere obvious.

- **`FacetView.get()` returns different types for different keys**: a `Facet` object for tensor
  views (q/k/v), a plain `int` for `n_heads`/`d_head`, and a bound method for `head`. Users
  need to know which keys return which type; the `menu()` output does not clarify this.
  A `type` or `kind` field in `FacetMenuItem` would help.

- **`Facet.value` is the access path, not `Facet.tensor`**: `fv.get('q').value` returns the
  tensor, but `fv.get('q').tensor` raises `AttributeError`. `Facet.__getattr__` proxies through
  to the underlying tensor, so `fv.get('q').shape` works, but `.value` is the explicit name.
  This is a mild ergonomic asymmetry vs. `Op.out`.

- **`FacetMenuItem.status='needs_capture'` for `pattern`/`scores`/`z` on SDPA models**:
  The attention pattern is not available when the model uses PyTorch's fused SDPA kernel.
  Users need `reconstruction_ready=True` (or `attn_implementation='eager'` in the HF model
  load). The menu detail message is verbose and informative, but `needs_capture` as a status
  string may be confused with missing `save=` predicates rather than a kernel-level limitation.

- **`Trace.input_preprocessor` is `None` when tracing with raw tensors** (not autorouted):
  `tl.trace(model, pixel_values)` sets `input_preprocessor=None`. Only `tl.trace(model,
  raw_text)` (via autoroute) or `tl.bridge.hf.trace_text/image` populates it. This is
  correct by design but may surprise users who call `bridge.hf.trace_image` and find
  the field missing after a direct tensor trace.

- **`autoroute.input` is not in `tl.__all__`**: `tl.autoroute` is not a top-level export.
  Users who want to inspect the detector registry must do `import torchlens.autoroute as ar`
  explicitly. There is no `tl.autoroute` shortcut.

- **`ModuleAccessor.keys()` returns call-qualified paths** (with `:1` suffixes for non-unique
  calls) mixed with unqualified ones. When iterating for filtering, users must guard with
  `":" not in key` to get the canonical path set. The presence of both qualified and unqualified
  keys in the same accessor is potentially confusing.

- **Vision bridge (`tl.bridge.hf.trace_image`)**: Not exercised here because the tiny ViT
  model was loaded with raw tensor input (no PIL). A real `trace_image` call requires PIL and
  a cached image processor. GAP: the image bridge path (`hf_image` autoroute detector,
  `ResolvedPreprocessing` with `source='hf_auto_image_processor'`) is NOT demonstrated in
  this notebook.